# Notebook séance 4 : Outils Python pour l'analyse EEG

PSY2007D — Laboratoire 1, automne 2026

Ce notebook introduit les outils Python que vous utiliserez pour analyser les données du projet (EEG pendant la marche, tâche lexicale). Il ne remplace pas les notebooks d'analyse : il sert à apprendre à les lire et à les modifier.

### Données utilisées dans ce notebook

Les données du projet 2026 ne sont pas encore enregistrées. Ce notebook utilise donc :

- un **jeu de données public** fourni avec MNE (*kiloword* : ERP de lecture de mots, Dufau et al., 2015), parties 2 et 3 ;
- des **données de marche simulées** (EEG et plateforme de force), partie 4 ;
- des **dossiers de sujets vides** et des **codes d'événements inventés**, partie 1, pour s'exercer à la structure.

Les notebooks d'analyse des données 2026 seront écrits quand les données seront disponibles.

### Remarque importante

Les notebooks d'analyse (ceux de 2025, dans la branche `2025` du dépôt, et ceux qui seront écrits pour 2026) suivent tous la même logique : on teste chaque étape **sur un seul sujet**, puis une **fonction** regroupe les étapes pour les appliquer à **tous les sujets**. Les résultats sont enregistrés dans un **tableau (CSV)**, qui sert ensuite aux statistiques et à l'apprentissage machine. Ce notebook reprend exactement ces briques.

### Objectifs pédagogiques

1. Vérifier que l'environnement de travail est fonctionnel.
2. Manipuler des chemins de fichiers, des listes et des dictionnaires (sujets, conditions, fenêtres temporelles).
3. Comprendre qu'un jeu d'epochs EEG est un tableau NumPy `(essais, canaux, temps)` et en extraire une amplitude moyenne.
4. Écrire une fonction et une boucle qui produisent un tableau de résultats avec pandas.
5. Aligner deux signaux enregistrés à des fréquences différentes (EEG et plateforme de force) et découper l'EEG autour des pas.

### Types de blocs

- **Bloc Type 1 — Méthode de base** : à exécuter et à lire.
- **Bloc Type 2 — À vous de compléter** : à modifier (les lignes à changer sont marquées `<---`).
- **Bloc Type 3 — Pour aller plus loin** : optionnel.

Dans Google Colab : `Fichier → Enregistrer une copie dans Drive` avant de commencer. En local : ouvrez ce fichier dans VS Code et sélectionnez l'environnement créé pendant l'installation.

## 0. Vérifier l'environnement

### Bloc Type 1 — Installation des paquets (Colab uniquement)

Colab contient déjà NumPy, pandas et Matplotlib, mais pas MNE. En local, ces paquets sont installés avec `requirements.txt` : ne lancez pas cette cellule.

In [ ]:
# -----------------------------------------------------------------------------
# Installation de MNE dans Colab (à ignorer en local)
# -----------------------------------------------------------------------------
import sys
if "google.colab" in sys.modules:          # vrai seulement dans Colab
    !pip install -q mne mne-bids

### Bloc Type 1 — Imports et versions

Si une ligne produit une erreur `ModuleNotFoundError`, le paquet correspondant n'est pas installé dans l'environnement actif.

In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux (les mêmes que dans les notebooks d'analyse)
# -----------------------------------------------------------------------------
from pathlib import Path          # manipulation des chemins de fichiers
import numpy as np                # calcul sur des tableaux
import pandas as pd               # tableaux de résultats
import matplotlib.pyplot as plt   # figures
import mne                        # analyse M/EEG

mne.set_log_level("WARNING")      # limite les messages de MNE
print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("pandas :", pd.__version__)
print("MNE    :", mne.__version__)

## 1. Chemins, listes et dictionnaires

Les données du projet seront rangées selon la convention **BIDS** : un dossier par sujet (`sub-01`, `sub-02`, ...), et les résultats dans `derivatives/`. Pour s'exercer, on crée ici une arborescence vide avec des sujets fictifs : aucun fichier EEG n'y est placé.

### Bloc Type 1 — Créer et parcourir une arborescence

In [ ]:
# -----------------------------------------------------------------------------
# Arborescence d'exemple : 6 sujets fictifs (dossiers vides) + un dossier de résultats
# -----------------------------------------------------------------------------
root = Path("donnees_seance4")                       # dossier racine (chemin relatif)
for i in range(1, 7):                                 # sujets 1 à 6
    (root / f"sub-{i:02d}" / "eeg").mkdir(parents=True, exist_ok=True)
deriv = root / "derivatives" / "seance4"              # dossier des résultats
deriv.mkdir(parents=True, exist_ok=True)

dossiers_sujets = sorted(root.glob("sub-*"))          # liste des dossiers sub-XX
subjects = [d.name.replace("sub-", "") for d in dossiers_sujets]
print("Racine    :", root.resolve())
print("Sujets    :", subjects)
print("Résultats :", deriv)

L'opérateur `/` assemble des morceaux de chemin, quel que soit le système (Windows, macOS, Linux). `f"sub-{i:02d}"` écrit le numéro sur deux chiffres (`01`, `02`, ...).

### Bloc Type 1 — Conditions et fenêtres temporelles

Les conditions sont décrites par un dictionnaire `event_id` (nom → code numérique du déclencheur). Avec MNE, un `/` dans le nom permet de sélectionner une partie des conditions (`"marche"`, `"mot"`, ...). Les fenêtres d'analyse des ERP sont aussi stockées dans un dictionnaire.

Les conditions ci-dessous (assis ou en marchant × mot ou pseudo-mot) et leurs codes sont **inventés** pour l'exercice : le plan réel de la tâche lexicale sera donné avec les données.

In [ ]:
# -----------------------------------------------------------------------------
# Conditions et codes inventés pour l'exercice (les vrais viendront du protocole)
# -----------------------------------------------------------------------------
event_id = {
    "assis/mot": 11,
    "assis/pseudo-mot": 12,
    "marche/mot": 21,
    "marche/pseudo-mot": 22,
}
fenetres = {                     # fenêtres temporelles en secondes après le stimulus
    "P200": (0.15, 0.25),
    "N400": (0.30, 0.50),
}

for nom, code_evt in event_id.items():
    print(f"{nom:<20} -> code {code_evt}")

conditions_marche = [nom for nom in event_id if nom.startswith("marche")]
print("Conditions de marche :", conditions_marche)

### Bloc Type 2 — À vous de compléter

1. Choisissez un sujet et construisez le chemin de son dossier `eeg`.
2. Ajoutez une fenêtre `"P600"` de 0.50 à 0.80 s.
3. Créez la liste des conditions qui concernent les **pseudo-mots**.

In [ ]:
# -----------------------------------------------------------------------------
# Sélection d'un sujet et ajout d'une fenêtre (modifiable)
# -----------------------------------------------------------------------------
subject = "03"                                  # <--- essayez un autre sujet
dossier_eeg = root / f"sub-{subject}" / "eeg"
print(dossier_eeg, "existe :", dossier_eeg.exists())

# fenetres["P600"] = ...                        # <--- à compléter
# conditions_pseudo = ...                       # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
fenetres["P600"] = (0.50, 0.80)
conditions_pseudo = [nom for nom in event_id if nom.endswith("pseudo-mot")]
print(fenetres)
print(conditions_pseudo)

## 2. L'EEG est un tableau NumPy

On utilise le jeu de données **kiloword** fourni avec MNE (Dufau et al., 2015) : des ERP en réponse à la lecture de 960 mots anglais, moyennés sur les participants, avec pour chaque mot des caractéristiques lexicales (fréquence, concrétude, nombre de lettres, ...). Ce ne sont **pas** les données du projet : c'est un jeu public qui permet de s'exercer dès maintenant sur une tâche de lecture de mots.

### Bloc Type 1 — Charger les epochs

Le premier chargement télécharge environ 25 Mo. Sans connexion, la cellule crée des données simulées de même structure pour que la suite du notebook fonctionne.

In [ ]:
# -----------------------------------------------------------------------------
# Chargement du jeu kiloword (ou données simulées de même format si hors ligne)
# -----------------------------------------------------------------------------
def charger_kiloword():
    try:
        chemin = mne.datasets.kiloword.data_path() / "kword_metadata-epo.fif"
        return mne.read_epochs(chemin, preload=True), "kiloword"
    except Exception as err:
        print("Téléchargement impossible, données simulées :", type(err).__name__)
        rng = np.random.default_rng(0)
        sfreq, times = 250.0, np.arange(-0.1, 0.924, 1 / 250)
        noms = ["Fz", "Cz", "Pz", "C3", "C4", "P3", "P4", "O1", "O2", "F3", "F4"]
        n = 960
        meta = pd.DataFrame({
            "WORD": [f"mot{i}" for i in range(n)],
            "Concreteness": rng.uniform(1, 7, n),
            "WordFrequency": rng.uniform(0, 4, n),
            "NumberOfLetters": rng.integers(3, 10, n).astype(float),
        })
        n400 = np.exp(-((times - 0.4) ** 2) / (2 * 0.07 ** 2))
        ampl = (-2 + 0.8 * meta["WordFrequency"].to_numpy())[:, None] * 1e-6
        data = ampl[:, None, :] * n400 + 1.5e-6 * rng.standard_normal((n, len(noms), len(times)))
        info = mne.create_info(noms, sfreq, "eeg")
        return mne.EpochsArray(data, info, tmin=times[0], metadata=meta), "simulé"

epochs, source = charger_kiloword()
print("Source :", source)
print(epochs)

### Bloc Type 1 — De l'objet MNE au tableau NumPy

`epochs.get_data()` renvoie un tableau à trois dimensions : `(essais, canaux, temps)`. Les valeurs sont en **volts**.

In [ ]:
# -----------------------------------------------------------------------------
# Structure des données
# -----------------------------------------------------------------------------
data = epochs.get_data()          # tableau (essais, canaux, temps), en volts
times = epochs.times              # temps de chaque échantillon (s)
print("Forme           :", data.shape)
print("Fréq. d'échant. :", epochs.info["sfreq"], "Hz")
print("Canaux          :", epochs.ch_names[:8], "...")
print("Temps           :", times[0], "à", times[-1], "s")

idx_pz = epochs.ch_names.index("Pz")                  # position du canal Pz
signal_pz = data[0, idx_pz, :]                        # essai 0, canal Pz, tous les temps
plt.figure(figsize=(8, 3))
plt.plot(times, signal_pz * 1e6)                      # conversion en microvolts
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.title(f"Essai 0, Pz : « {epochs.metadata['WORD'].iloc[0]} »")
plt.show()

### Bloc Type 1 — Sélectionner une fenêtre temporelle

Un **masque** est un tableau de `True`/`False` de la même longueur que `times`. Il sert à ne garder que les échantillons d'une fenêtre (même principe que la fonction `_window_mask` des notebooks d'analyse 2025).

In [ ]:
# -----------------------------------------------------------------------------
# Amplitude moyenne dans la fenêtre N400, canal Pz, pour tous les essais
# -----------------------------------------------------------------------------
tmin, tmax = fenetres["N400"]
masque = (times >= tmin) & (times <= tmax)
print("Échantillons dans la fenêtre :", masque.sum())

n400_pz = data[:, idx_pz, masque].mean(axis=1) * 1e6  # une valeur par essai, en µV
print("Forme :", n400_pz.shape)
print("Moyenne sur les mots : %.2f µV" % n400_pz.mean())

### Bloc Type 2 — À vous de compléter

Calculez la même amplitude moyenne pour la fenêtre **P200** au canal **Cz**.

In [ ]:
# -----------------------------------------------------------------------------
# P200 au canal Cz (modifiable)
# -----------------------------------------------------------------------------
# idx_cz = ...                                   # <--- à compléter
# masque_p200 = ...                              # <--- à compléter
# p200_cz = ...                                  # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
idx_cz = epochs.ch_names.index("Cz")
tmin, tmax = fenetres["P200"]
masque_p200 = (times >= tmin) & (times <= tmax)
p200_cz = data[:, idx_cz, masque_p200].mean(axis=1) * 1e6
print("P200 moyenne à Cz : %.2f µV" % p200_cz.mean())

## 3. Fonctions, boucles et tableau de résultats

On regroupe le calcul précédent dans une **fonction**, puis on l'applique à plusieurs canaux et fenêtres avec une **boucle**. Chaque résultat devient une ligne (un dictionnaire) ; la liste de lignes devient un `DataFrame` pandas. C'est la structure de la fonction `process_subject_features` des notebooks d'analyse 2025.

### Bloc Type 1 — Fonction d'amplitude moyenne

In [ ]:
# -----------------------------------------------------------------------------
# Fonction réutilisable : amplitude moyenne (µV) par essai
# -----------------------------------------------------------------------------
def amplitude_moyenne(epochs, canal, tmin, tmax):
    # Renvoie un tableau (n_essais,) : moyenne entre tmin et tmax au canal donné, en µV.
    idx = epochs.ch_names.index(canal)
    masque = (epochs.times >= tmin) & (epochs.times <= tmax)
    return epochs.get_data()[:, idx, masque].mean(axis=1) * 1e6

print(amplitude_moyenne(epochs, "Pz", 0.30, 0.50)[:5])

### Bloc Type 1 — Boucle sur canaux et fenêtres

In [ ]:
# -----------------------------------------------------------------------------
# Une ligne par (canal, fenêtre) : moyenne sur tous les mots
# -----------------------------------------------------------------------------
canaux = ["Fz", "Cz", "Pz"]
lignes = []
for canal in canaux:
    for nom_fenetre, (tmin, tmax) in fenetres.items():
        valeurs = amplitude_moyenne(epochs, canal, tmin, tmax)
        lignes.append({"canal": canal, "fenetre": nom_fenetre, "amplitude_uV": valeurs.mean()})

resume = pd.DataFrame(lignes)
resume

### Bloc Type 1 — Tableau par essai avec les métadonnées

`epochs.metadata` est déjà un `DataFrame` : une ligne par mot, une colonne par caractéristique. On y ajoute l'amplitude N400, puis on compare les mots **peu fréquents** et **fréquents**. Selon la littérature, les mots peu fréquents produisent une N400 plus ample (plus négative).

In [ ]:
# -----------------------------------------------------------------------------
# Tableau par mot : métadonnées + amplitude N400 à Pz
# -----------------------------------------------------------------------------
df = epochs.metadata.copy()
df["N400_Pz_uV"] = amplitude_moyenne(epochs, "Pz", *fenetres["N400"])
df["frequence"] = pd.qcut(df["WordFrequency"], 2, labels=["faible", "élevée"])  # coupe à la médiane

print(df[["WORD", "WordFrequency", "frequence", "N400_Pz_uV"]].head())
print()
print(df.groupby("frequence", observed=True)["N400_Pz_uV"].agg(["mean", "std", "count"]))

In [ ]:
# -----------------------------------------------------------------------------
# Enregistrer le tableau (les notebooks d'analyse enregistrent leurs résultats ainsi)
# -----------------------------------------------------------------------------
fichier_csv = deriv / "n400_par_mot.csv"
df.to_csv(fichier_csv, index=False)
relu = pd.read_csv(fichier_csv)
print(fichier_csv, "->", relu.shape, "lignes x colonnes")

### Bloc Type 1 — ERP moyen par groupe de mots

On sélectionne les essais avec une requête sur les métadonnées, puis on moyenne sur les essais (axe 0).

In [ ]:
# -----------------------------------------------------------------------------
# ERP à Pz pour les mots peu fréquents vs fréquents
# -----------------------------------------------------------------------------
mediane = df["WordFrequency"].median()
plt.figure(figsize=(8, 3))
for etiquette, requete in [("faible", f"WordFrequency < {mediane}"), ("élevée", f"WordFrequency >= {mediane}")]:
    erp = epochs[requete].get_data()[:, idx_pz, :].mean(axis=0) * 1e6
    plt.plot(times, erp, label=f"fréquence {etiquette}")
plt.axvspan(*fenetres["N400"], color="0.9")
plt.axvline(0, color="k", lw=0.8)
plt.gca().invert_yaxis()                       # convention ERP : négatif vers le haut
plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.title("Pz"); plt.legend()
plt.show()

### Bloc Type 2 — À vous de compléter

Refaites la comparaison avec la **concrétude** (`Concreteness`) au lieu de la fréquence : colonne de groupes, `groupby`, puis la figure.

In [ ]:
# -----------------------------------------------------------------------------
# Comparaison selon la concrétude (modifiable)
# -----------------------------------------------------------------------------
# df["concretude"] = ...                          # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
df["concretude"] = pd.qcut(df["Concreteness"], 2, labels=["abstrait", "concret"])
print(df.groupby("concretude", observed=True)["N400_Pz_uV"].agg(["mean", "std", "count"]))
med = df["Concreteness"].median()
plt.figure(figsize=(8, 3))
for etiquette, requete in [("abstrait", f"Concreteness < {med}"), ("concret", f"Concreteness >= {med}")]:
    plt.plot(times, epochs[requete].get_data()[:, idx_pz, :].mean(axis=0) * 1e6, label=etiquette)
plt.gca().invert_yaxis(); plt.legend(); plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.show()

### Bloc Type 3 — Pour aller plus loin

La fréquence est une variable continue. Tracez l'amplitude N400 en fonction de `WordFrequency` (nuage de points) et calculez la corrélation (`df["WordFrequency"].corr(df["N400_Pz_uV"])`).

## 4. Données de marche : aligner l'EEG et la plateforme de force

Pendant la tâche de marche, plusieurs systèmes enregistrent en même temps (EEG, capture du mouvement, EMG, plateformes de force), chacun avec **sa propre fréquence d'échantillonnage** et **son propre début d'enregistrement**. Pour analyser l'EEG autour de chaque pas, il faut :

1. détecter les contacts du talon (*heel strikes*) dans le signal de force ;
2. convertir ces instants dans le temps de l'EEG ;
3. découper l'EEG en epochs autour de ces instants.

Les données ci-dessous sont **simulées** : aucune donnée de marche n'a encore été enregistrée. Les fréquences d'échantillonnage, la durée et le décalage entre les deux systèmes sont des valeurs d'exemple ; le format des vraies données sera présenté quand elles seront disponibles.

### Bloc Type 1 — Simuler les deux enregistrements

In [ ]:
# -----------------------------------------------------------------------------
# Simulation : EEG à 300 Hz, force verticale à 1000 Hz, débuts décalés
# -----------------------------------------------------------------------------
rng = np.random.default_rng(1)
duree = 60.0                                   # secondes de marche

# Instants des contacts du talon gauche (une foulée toutes les 1,1 s environ)
pas = np.cumsum(rng.normal(1.10, 0.03, 60))
pas = pas[pas < duree - 1]

# Force verticale sous le pied gauche : 1000 Hz, démarre 2,0 s après l'EEG (horloge commune)
fs_force, t0_force = 1000, 2.0
t_force = t0_force + np.arange(0, duree, 1 / fs_force)
force = np.zeros_like(t_force)
for t_pas in pas + t0_force:                   # une bosse de force par appui (0,65 s)
    phase = (t_force - t_pas) / 0.65
    appui = (phase >= 0) & (phase <= 1)
    force[appui] += 700 * np.sin(np.pi * phase[appui])
force += rng.normal(0, 5, force.size)

# EEG : 300 Hz, 4 canaux, bruit + alpha + artefact lié à chaque pas
fs_eeg, t0_eeg = 300, 0.0
t_eeg = t0_eeg + np.arange(0, duree + t0_force, 1 / fs_eeg)
noms_eeg = ["Fz", "Cz", "Pz", "Oz"]
eeg = rng.normal(0, 5e-6, (len(noms_eeg), t_eeg.size))
phase_alpha = np.cumsum(rng.normal(0, 0.05, t_eeg.size))       # phase qui dérive lentement
eeg += 4e-6 * np.sin(2 * np.pi * 10 * t_eeg + phase_alpha)      # rythme alpha
for t_pas in pas + t0_force:
    apres = t_eeg - t_pas
    fen = (apres >= 0) & (apres < 0.3)
    eeg[:, fen] += 15e-6 * np.exp(-apres[fen] / 0.05) * np.array([[1.0], [0.6], [0.4], [0.8]])

print(f"EEG   : {eeg.shape}, {fs_eeg} Hz, début à {t0_eeg} s")
print(f"Force : {force.shape}, {fs_force} Hz, début à {t0_force} s")

In [ ]:
# -----------------------------------------------------------------------------
# Visualiser 5 s des deux signaux sur un axe de temps commun
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t_force, force); axes[0].set_ylabel("Force (N)")
axes[1].plot(t_eeg, eeg[0] * 1e6); axes[1].set_ylabel("Fz (µV)")
axes[1].set_xlim(10, 15); axes[1].set_xlabel("Temps (s)")
plt.show()

### Bloc Type 1 — Détecter les contacts du talon

Un contact correspond au moment où la force dépasse un seuil en montant. On compare chaque échantillon au précédent avec `np.diff`, puis on élimine les détections multiples causées par le bruit autour du seuil.

In [ ]:
# -----------------------------------------------------------------------------
# Détection par franchissement de seuil
# -----------------------------------------------------------------------------
seuil = 50                                             # newtons
au_dessus = force > seuil                              # True pendant l'appui
debuts = np.where(np.diff(au_dessus.astype(int)) == 1)[0] + 1
t_contacts = t_force[debuts]                           # instants en secondes (horloge commune)

# Le bruit fait franchir le seuil plusieurs fois au même appui : on garde une détection par foulée
def une_par_foulee(instants, ecart_min=0.8):
    # Garde un instant seulement s'il est à plus de ecart_min secondes du dernier instant gardé.
    gardes = []
    for t in instants:
        if len(gardes) == 0 or t - gardes[-1] > ecart_min:
            gardes.append(t)
    return np.array(gardes)

print(len(t_contacts), "franchissements bruts")
t_contacts = une_par_foulee(t_contacts)
print(len(t_contacts), "contacts détectés ; premiers :", np.round(t_contacts[:4], 3))

### Bloc Type 1 — Créer les epochs EEG autour des pas avec MNE

On crée un objet `Raw` à partir du tableau EEG, on ajoute une **annotation** à chaque contact (converti dans le temps de l'EEG), puis on utilise les mêmes fonctions que dans les notebooks d'analyse : `events_from_annotations` et `Epochs`.

In [ ]:
# -----------------------------------------------------------------------------
# Tableau -> Raw -> annotations -> événements -> epochs -> moyenne
# -----------------------------------------------------------------------------
info = mne.create_info(noms_eeg, fs_eeg, "eeg")
raw = mne.io.RawArray(eeg, info)

onsets = t_contacts - t0_eeg                           # temps relatif au début de l'EEG
raw.set_annotations(mne.Annotations(onset=onsets, duration=0, description="heel_strike"))

events, ids = mne.events_from_annotations(raw)
epochs_pas = mne.Epochs(raw, events, ids, tmin=-0.2, tmax=0.5, baseline=(-0.2, 0), preload=True)
print(epochs_pas)

moyenne_pas = epochs_pas.average()
moyenne_pas.plot(titles=dict(eeg="EEG moyen autour du contact du talon"));

Le signal moyen montre une déflexion au moment du contact du talon : c'est un **artefact mécanique**, pas une réponse cérébrale. Pendant la marche, ce type d'artefact s'ajoute aux réponses de la tâche lexicale ; le prétraitement (séance du 14 octobre) sert en partie à le réduire.

### Bloc Type 2 — À vous de compléter

1. Mettez le seuil à 400 N. Le nombre de contacts change-t-il ? Et l'instant détecté ? Regardez la latence de l'artefact dans la moyenne.
2. Oubliez volontairement la conversion `- t0_eeg` (remplacez `t0_eeg` par `2.0` dans le calcul des `onsets`). Que devient la moyenne ? Qu'est-ce que cela montre sur l'alignement des signaux ?

In [ ]:
# -----------------------------------------------------------------------------
# Effet du seuil et de l'alignement (modifiable)
# -----------------------------------------------------------------------------
seuil_test = 50                                        # <--- essayez 400
decalage_test = t0_eeg                                 # <--- essayez 2.0

debuts_test = np.where(np.diff((force > seuil_test).astype(int)) == 1)[0] + 1
onsets_test = une_par_foulee(t_force[debuts_test]) - decalage_test
onsets_test = onsets_test[(onsets_test > 0.2) & (onsets_test < raw.times[-1] - 0.5)]
raw_test = raw.copy().set_annotations(mne.Annotations(onsets_test, 0, "heel_strike"))
ev, ev_id = mne.events_from_annotations(raw_test)
mne.Epochs(raw_test, ev, ev_id, tmin=-0.2, tmax=0.5, baseline=(-0.2, 0), preload=True).average().plot();
print(len(onsets_test), "contacts utilisés")

### Bloc Type 3 — Pour aller plus loin

Les deux signaux n'ont pas la même fréquence d'échantillonnage. Rééchantillonnez la force à 300 Hz avec `np.interp(t_eeg, t_force, force)` et tracez-la sur le même axe que l'EEG.

## 5. Récapitulatif

| Brique vue aujourd'hui | À quoi elle servira avec les données du projet |
|---|---|
| `Path`, dossiers `sub-XX`, `derivatives/` | retrouver les fichiers de chaque sujet et ranger les résultats |
| dictionnaires `event_id`, `fenetres` | définition des conditions et des fenêtres ERP |
| tableau `(essais, canaux, temps)`, masque temporel | calcul des amplitudes et latences |
| fonction + boucle + `DataFrame` + `to_csv` | fonction finale appliquée à tous les sujets |
| `Raw`, annotations, `events_from_annotations`, `Epochs` | découpage des données de la tâche lexicale et de la marche |

### Prochaines étapes

- **7 octobre** : introduction à MNE (visualisation, filtrage, spectres). Venez avec un environnement fonctionnel.
- **14 octobre** : cours de traitement des données (prétraitement, artefacts).
- Avant le 7 octobre : terminez les blocs Type 2 de ce notebook et vérifiez votre installation avec `verifier_installation.py`.